# Step 3 — Raw data-quality audit of the anime and user tables

**Purpose.** Step 1 compared the raw data with the author's description. This stage looks inside the columns that the research questions will depend on, and fixes the `null` username problem found in step 1.

**How to run on Kaggle.** Add the *MyAnimeList Dataset* as input and run all cells. Parts 1–3 only read the two small tables and take seconds. Part 4 reads two columns of the 80M-row list table and takes a few minutes; set `RUN_LISTS = False` in the setup cell to skip it.

**Reading the output.** Each part ends with an *Observations* cell marked TODO. Fill it in after the run, writing what you see before interpreting it.

| Part | Question |
|---|---|
| 1 | Which real values does pandas silently turn into `NaN`? |
| 2 | Anime table: placeholders, impossible values, how reliable is `score`, what can a pre-airing model use? |
| 3 | User table: placeholders, impossible ages and dates, suspicious aggregates |
| 4 | Do the per-user counts in `UserList` agree with the rows in `UserAnimeList`? |
| 5 | The users with no row in the list table: do they have other values, which would mean the list table is missing them? |
| 6 | Clarity checks: numbers that the first run did not print, and hypotheses that were never tested |

**Where this step sits.** Step 3 of the exploration order (see [docs/exploration_order.md](../docs/exploration_order.md)). This notebook only **finds and counts** problems. It does not clean anything and it does not compare columns statistically. Cleaning rules are decided in step 4, from the evidence found here.

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Works both on Kaggle and inside the repo (with the data placed in data/raw/)
KAGGLE_INPUT = Path("/kaggle/input")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "environment.yml").exists()), None)
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
elif ROOT is not None:
    DATA_DIR = ROOT / "data" / "raw"
else:
    raise RuntimeError("Data not found: run on Kaggle or inside the repository.")

paths = {p.name: p for p in DATA_DIR.rglob("*.csv")}
RUN_LISTS = True  # False skips part 4, which reads 80M rows

print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)
print("Number of CSV files:", len(paths))

Python: 3.13.18 | pandas: 2.3.3
Number of CSV files: 9


## 1. Real values that pandas turns into `NaN`

By default pandas reads strings such as `null`, `NA` or `None` as missing. In step 1 this made the username `null` disappear. Here every cell of the two small tables is read as raw text, and the cells that pandas would have turned into `NaN` are counted per column. Both tables are then loaded again with only the empty string treated as missing.

### 1a. First look at the values, before counting the hidden missing words (NEW, added after the first run)

**Why this comes first.** Below, we count the words that pandas hides (`null`, `N/A`, `None`, ...) with a list of words that we wrote ourselves. In part 2 we also count placeholders with a list we wrote (`Unknown`, `Not available`, `None`, `?`, `0000-00-00`). Both lists come from what we happened to see, so they can miss other words, and they only find a cell that **equals** one of the words. A cell such as `Fridays at Unknown` is not found. So before any counting, we let the data show us which placeholder-like values exist.

**What the cells do.** The first one loads the two small tables with the safe load. For every text column of the anime table and the user table it counts three things, and prints the values behind each count:
- **Exact candidates:** a cell that is only a placeholder-like word, a symbol or a zero (`?`, `-`, `0`, `none`, `null`, `n/a`, `unknown`, `tba`, `tbd`, `undefined`, `not available`, `0000-00-00`, spaces only), in any capital letters.
- **Partial candidates:** a longer text that contains `unknown`, `not available`, `n/a`, `tba` or `tbd` (for example `Fridays at Unknown`).
- **Very short texts:** two characters or fewer, which are often a typo or a symbol.

The second cell prints the five most frequent values of every text column, so that we can read them with our own eyes: a placeholder is usually one of the most frequent values of its column.

**Limits.** This still looks for patterns that we think of. A placeholder written in an unexpected way (`idk`, `nothing`, `Earth` in `location`) can only be found by reading the most frequent values. The list table (80 million rows) is not scanned here; its text columns are `my_tags` (free text) and the two dates, which part 3 of notebook 02 already inspected.

**Read every hit before calling it a placeholder.** The scan only finds candidates. For example, the username `null` is a real user, so it will appear as an exact candidate in `username` although it is not a placeholder.

**What we do with the result.** If the output shows words that are not in our lists, we add them to the lists below and in part 2 before counting.


In [2]:
# NEW: placeholder-like cells in the text columns of the anime and user tables
import re

na_kwargs = dict(keep_default_na=False, na_values=[""])  # only an empty cell is missing
anime = pd.read_csv(paths["AnimeList.csv"], **na_kwargs)
users = pd.read_csv(paths["UserList.csv"], **na_kwargs)

CANDIDATE = r"^\s*(?:|\?+|-+|\.+|_+|0|none|null|nan|n/?a|unknown|unk|tba|tbd|undefined|not available|not specified|no data|0000-00-00)\s*$"
PARTIAL = r"unknown|not available|n/a|tba|tbd"

def scan_text(df, label):
    print("=====", label)
    rows, details = [], {}
    for col in df.select_dtypes(exclude=["number", "bool"]).columns:
        s = df[col].dropna().astype(str)
        exact = s[s.str.match(CANDIDATE, flags=re.IGNORECASE)]
        partial = s[s.str.contains(PARTIAL, flags=re.IGNORECASE, regex=True)]
        partial = partial[~partial.index.isin(exact.index)]
        short = s[(s.str.len() <= 2) & ~s.index.isin(exact.index)]
        rows.append({"column": col, "non_missing": len(s), "exact": len(exact), "partial": len(partial), "short": len(short)})
        details[col] = (exact, partial, short)
    table = pd.DataFrame(rows).set_index("column")
    hits = table[(table[["exact", "partial", "short"]] > 0).any(axis=1)]
    print(hits.sort_values("exact", ascending=False).to_string())
    for col in hits.index:
        exact, partial, short = details[col]
        print(f"\n{col}")
        if len(exact): print("  exact  :", exact.value_counts().head(6).to_dict())
        if len(partial): print("  partial:", partial.value_counts().head(6).to_dict())
        if len(short): print("  short  :", short.value_counts().head(6).to_dict())

scan_text(anime, "anime table")
print()
scan_text(users, "user table")


===== anime table
                non_missing  exact  partial  short
column                                            
source                14478   4210        0      0
broadcast              4271   2241       78      0
rating                14478    544        0      0
duration              14478    405        0      0
aired_string          14478    223        0      0
type                  14478     30        0   4271
title_synonyms         8937      0       16     11
title_japanese        14443      0        1    165
title                 14478      0        2     12
title_english          5724      0        9      3
background             1057      0        2      0
related               14478      0        1   4515
producer               8288      0        9      6
studio                 8544      0        0      3
opening_theme         14478      0        4   9784
ending_theme          14478      0        5   8807

title
  partial: {'The Epic Of ZektBach': 1, 'GetBackers': 1}
 

In [3]:
# NEW: the five most frequent values of every text column (read them: placeholders are often among them)
def top_values(df, label, n=5):
    print("=====", label)
    for col in df.select_dtypes(exclude=["number", "bool"]).columns:
        counts = df[col].value_counts().head(n)
        print(f"{col}: " + " | ".join(f"{str(v)[:30]!r} {c:,}" for v, c in counts.items()))

top_values(anime, "anime table")
print()
top_values(users, "user table")


===== anime table
title: 'Hinamatsuri' 2 | 'Mushishi' 1 | 'Macross Dynamite 7' 1 | 'Ashita no Joe (Movie)' 1 | 'Decorator' 1
title_english: 'Cyborg 009' 4 | 'Digimon: The Movie' 3 | 'Big Windup!' 3 | 'Fight! Kickers' 3 | 'The Heroic Legend of Arslan' 3
title_japanese: 'ゲゲゲの鬼太郎' 6 | 'うっかりペネロペ' 4 | '아기공룡 둘리' 4 | '宇宙の騎士テッカマンブレード' 4 | 'ぼのぼの' 4
title_synonyms: 'Minna no Uta' 189 | 'Irodorimidori' 23 | 'Sekai Meisaku Gekijou' 5 | 'Doutoku Meisaku Series' 5 | 'Backkom, Bernard' 4
image_url: 'https://myanimelist.cdn-dena.c' 1 | 'https://myanimelist.cdn-dena.c' 1 | 'https://myanimelist.cdn-dena.c' 1 | 'https://myanimelist.cdn-dena.c' 1 | 'https://myanimelist.cdn-dena.c' 1
type: 'TV' 4,271 | 'OVA' 3,612 | 'Movie' 2,624 | 'Special' 1,948 | 'ONA' 1,144
source: 'Unknown' 4,210 | 'Original' 3,368 | 'Manga' 3,120 | 'Visual novel' 888 | 'Game' 607
status: 'Finished Airing' 13,791 | 'Currently Airing' 351 | 'Not yet aired' 336
aired_string: 'Not available' 223 | '2005' 47 | '2004' 43 | 'Jul, 2018 to ?'

### Observations for the first look

**TODO (fill in after the run):**
- Which columns hold exact candidates, partial candidates or very short texts, and how many? Is any word in the output missing from the lists of part 1 and part 2?
- For `broadcast`: how many cells contain `Unknown` inside a longer text, in addition to the exact ones?
- In the five most frequent values of each column, which ones look like a placeholder (for example in `location`, `title_english`, `studio`, `producer`)?


### 1b. Four checks on the guesses of part 1a (NEW, **not run yet**)

**Why.** The first look (1a) produced four guesses that we wrote down as hypotheses. Each one can be turned into a number with the two small tables that are already loaded, so we test them here before we rely on them.

| # | Guess from 1a | What the cell counts | What the result would mean |
|---|---|---|---|
| 1 | Some titles still contain HTML codes such as `&#039;` | The cells with a code like `&#039;` or `&amp;`, per text column, and the most frequent codes | If many cells are affected, titles must be decoded (`html.unescape`) before matching or counting words |
| 2 | The 11th may be an automatic day in start dates | The number of start dates (`aired`) for each day of the month, 1 to 31 | If day 11 is far above the other days, it is a default day and those dates are only month-accurate |
| 3 | January 1 may be a default birth day | The share of January 1 among all birth dates, the most frequent month-day values, and the share of January 1 for some birth years | If January 1 is far above 1/365 (0.27%), only the year of those users can be trusted |
| 4 | `aired` is empty when the date is only partial, and `aired_string` still holds it | For the anime with an empty `aired`: what kind of text `aired_string` holds (full date, month and year, year only, empty). The same for the anime with status `Not yet aired` and an empty `aired` (the 147 of notebook 02) | If `aired_string` holds a year or a month, the release year must be read from `aired_string`, and "not yet aired without a start date" does not mean "without any date" |

**Limits.** The cell only describes what is in the two small tables. It cannot tell us why a default day exists, only that the counts look like one.


In [4]:
# NEW: four checks on the guesses of part 1a (they only read the two small tables)
import ast, html

# 1. HTML codes in the text columns
ENTITY = r"&#?\w+;"
print("1. Cells with an HTML code (for example &#039; or &amp;)")
for label, df in (("anime", anime), ("users", users)):
    for col in df.select_dtypes(exclude=["number", "bool"]).columns:
        s = df[col].dropna().astype(str)
        hit = s[s.str.contains(ENTITY, regex=True)]
        if len(hit):
            codes = hit.str.findall(ENTITY).explode().value_counts().head(5).to_dict()
            print(f"   {label}.{col}: {len(hit):,} of {len(s):,} cells ({len(hit) / len(s):.2%}) | codes: {codes}")
            print(f"      example: {hit.iloc[0][:60]!r} -> {html.unescape(hit.iloc[0])[:60]!r}")

# 2. Day of the month of the start date in `aired`
def aired_from(t):
    try:
        return pd.to_datetime(ast.literal_eval(t).get("from"), utc=True, errors="coerce")
    except Exception:
        return pd.NaT

start = pd.to_datetime(anime["aired"].map(aired_from), utc=True)
day = start.dt.day.dropna().astype(int)
per_day = day.value_counts().reindex(range(1, 32), fill_value=0)
print()
print(f"2. Day of the month of the start date: {len(day):,} start dates, mean per day {len(day) / 31:.0f}, median day count {per_day.median():.0f}")
print("   count per day:", per_day.to_dict())
print("   top 3 days:", per_day.sort_values(ascending=False).head(3).to_dict())
d11 = start[start.dt.day == 11]
print("   start dates on the 11th, most frequent year-months:", d11.dt.strftime("%Y-%m").value_counts().head(5).to_dict())

# 3. January 1 among the birth dates
bd = pd.to_datetime(users["birth_date"], errors="coerce").dropna()
month_day = bd.dt.strftime("%m-%d")
share_jan1 = (month_day == "01-01").mean()
print()
print(f"3. Birth dates: {len(bd):,} parsed. January 1: {(month_day == '01-01').sum():,} users = {share_jan1:.2%} (uniform days would give {1 / 365:.2%})")
print("   most frequent month-day values:", month_day.value_counts().head(5).to_dict())
by_year = bd[bd.dt.year.between(1985, 2000)]
jan1_by_year = (by_year.dt.strftime("%m-%d") == "01-01").groupby(by_year.dt.year).mean().round(3)
print("   share of January 1 per birth year, 1985-2000:", jan1_by_year.to_dict())

# 4. `aired` against `aired_string`
def kind(t):
    if pd.isna(t) or str(t).strip().lower() in ("", "not available"): return "empty"
    if re.search(r"[A-Z][a-z]{2} \d{1,2}, \d{4}", t): return "full date"
    if re.search(r"[A-Z][a-z]{2}, \d{4}", t): return "month and year"
    if re.search(r"\b\d{4}\b", t): return "year only"
    return "other"

string_kind = anime["aired_string"].map(kind)
no_start = start.isna()
print()
print(f"4. `aired` has no start date for {int(no_start.sum()):,} anime. What `aired_string` holds for them:")
print("  ", string_kind[no_start].value_counts().to_dict())
print("   other texts (first 8):", anime.loc[no_start & (string_kind == "other"), "aired_string"].value_counts().head(8).to_dict())
print("   `aired` has a start date but `aired_string` is empty:", int((~no_start & (string_kind == "empty")).sum()))
unaired = no_start & (anime["status"] == "Not yet aired")
print(f"   status `Not yet aired` and no start date in `aired`: {int(unaired.sum()):,}. What `aired_string` holds:")
print("  ", string_kind[unaired].value_counts().to_dict())
print("   most frequent texts:", anime.loc[unaired, "aired_string"].value_counts().head(6).to_dict())


1. Cells with an HTML code (for example &#039; or &amp;)
   anime.title: 202 of 14,478 cells (1.40%) | codes: {'&#039;': 213}
      example: 'Dae Jang Geum: Jang Geum&#039;s Dream' -> "Dae Jang Geum: Jang Geum's Dream"
   anime.title_english: 409 of 5,724 cells (7.15%) | codes: {'&#039;': 426}
      example: 'Sekai Ichi Hatsukoi - World&#039;s Greatest First Love 2' -> "Sekai Ichi Hatsukoi - World's Greatest First Love 2"
   anime.title_japanese: 76 of 14,443 cells (0.53%) | codes: {'&#039;': 79}
      example: '電脳戦隊ヴギィ&#039;ズ★エンジェル外伝 進め! スーパー★エンジェルス!' -> "電脳戦隊ヴギィ'ズ★エンジェル外伝 進め! スーパー★エンジェルス!"
   anime.title_synonyms: 739 of 8,937 cells (8.27%) | codes: {'&#039;': 898}
      example: 'Sekai-ichi Hatsukoi 2, Sekai&#039;ichi Hatsukoi 2' -> "Sekai-ichi Hatsukoi 2, Sekai'ichi Hatsukoi 2"
   anime.background: 316 of 1,057 cells (29.90%) | codes: {'&#039;': 378, '&rsquo;': 26, '&rdquo;': 11, '&mdash;': 11, '&ndash;': 8}
      example: 'Bleach adapts the first 54 volumes of Tite Kubo&#039;s man

### Observations for the four checks

**TODO (fill in after the run):**
- Check 1: how many cells have an HTML code, in which columns, and which codes are the most frequent?
- Check 2: is day 11 clearly above the other days? Are the dates on the 11th concentrated in a few year-months?
- Check 3: what is the share of January 1 (compare with 0.27%)? Is it the same for every birth year?
- Check 4: for the anime without an `aired` start date, how many have a full date, a month and year, a year only, or nothing in `aired_string`? What does this say about the 147 `Not yet aired` anime of notebook 02?
- Update the rows 20 to 22 of the summary of step 3 with the counts.


In [5]:
NA_TOKENS = {"null", "NULL", "NA", "N/A", "n/a", "NaN", "nan", "None", "#N/A", "#NA",
             "-NaN", "-nan", "<NA>", "1.#IND", "1.#QNAN", "-1.#IND", "-1.#QNAN", "#N/A N/A"}

for file_name in ["AnimeList.csv", "UserList.csv"]:
    text = pd.read_csv(paths[file_name], dtype=str, keep_default_na=False)
    hits = {col: int(text[col].isin(NA_TOKENS).sum()) for col in text.columns}
    hits = {col: count for col, count in hits.items() if count}
    print(file_name, "-> columns with hidden NA tokens:", hits or "none")
    for col in hits:
        print("   ", col, text.loc[text[col].isin(NA_TOKENS), col].value_counts().head(5).to_dict())
    del text

# Safe load: only the empty string is missing
na_kwargs = dict(keep_default_na=False, na_values=[""])
anime = pd.read_csv(paths["AnimeList.csv"], **na_kwargs)
users = pd.read_csv(paths["UserList.csv"], **na_kwargs)
print("\nanime:", anime.shape, "| users:", users.shape)
print("username NaN:", users.username.isna().sum(),
      "| username unique:", users.username.is_unique,
      "| distinct usernames:", f"{users.username.nunique():,}", "(description: 302,675)")
assert users.username.is_unique

AnimeList.csv -> columns with hidden NA tokens: {'rating': 544}
    rating {'None': 544}
UserList.csv -> columns with hidden NA tokens: {'username': 1, 'location': 26}
    username {'null': 1}
    location {'N/A': 15, 'n/a': 7, 'NA': 3, 'None': 1}

anime: (14478, 31) | users: (302675, 17)
username NaN: 0 | username unique: True | distinct usernames: 302,675 (description: 302,675)


### Observations for part 1

**TODO (fill in after the run):** which columns lost real values, how many, and which values. Does `users` now have 302,675 distinct usernames?

## 2. Anime table

Columns that matter for the candidate questions: `type`, `source`, `episodes`, `duration`, `aired_string`, `genre`, `status`, `rating` (age rating), `related` (sequels, prequels), and the audience columns `score`, `scored_by`, `rank`, `popularity`, `members`, `favorites`.

For a model that predicts the score **before an anime airs**, the audience columns (`score`, `scored_by`, `rank`, `popularity`, `members`, `favorites`) are only known after airing, so they must be excluded as leakage.

In [6]:
def missing_overview(df):
    return pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
    }).sort_values("missing_pct", ascending=False)


print(anime.dtypes.value_counts().to_dict())
print(missing_overview(anime).head(12))

# Placeholder values that are not NaN but also carry no information
PLACEHOLDERS = {"", "Unknown", "Not available", "None", "?", "0000-00-00"}
text_cols = anime.select_dtypes(exclude="number").columns
placeholder_counts = pd.Series({c: int(anime[c].isin(PLACEHOLDERS).sum()) for c in text_cols})
print("\nplaceholder values per column:")
print(placeholder_counts[placeholder_counts > 0].sort_values(ascending=False))

print("\nduplicated anime_id:", anime.anime_id.duplicated().sum())

{dtype('O'): 22, dtype('int64'): 6, dtype('float64'): 2, dtype('bool'): 1}
                missing_count  missing_pct
background              13421        92.70
licensor                11105        76.70
premiered               10382        71.71
broadcast               10207        70.50
title_english            8754        60.46
producer                 6190        42.75
studio                   5934        40.99
title_synonyms           5541        38.27
rank                     1574        10.87
image_url                  96         0.66
genre                      64         0.44
title_japanese             35         0.24

placeholder values per column:
source          4210
broadcast       2241
rating           544
duration         405
aired_string     223
type              30
dtype: int64

duplicated anime_id: 0


In [7]:
# Numeric columns: are the ranges plausible?
num_cols = ["score", "scored_by", "rank", "popularity", "members", "favorites", "episodes"]
print(anime[num_cols].describe(percentiles=[.01, .25, .5, .75, .99]).T.round(2))

# How reliable is the published score?
print("\nscore == 0:", (anime.score == 0).sum(),
      "| scored_by == 0:", (anime.scored_by == 0).sum(),
      "| both:", ((anime.score == 0) & (anime.scored_by == 0)).sum())
for k in (10, 50, 100, 1000):
    share = (anime.scored_by < k).mean()
    print(f"anime with scored_by < {k:>4}: {(anime.scored_by < k).sum():>6,} ({share:.1%})")
print("episodes == 0:", (anime.episodes == 0).sum())

              count      mean       std  min      1%      25%      50%  \
score       14478.0      6.14      1.46  0.0    0.00     5.55     6.37   
scored_by   14478.0  11460.03  43105.19  0.0    0.00    46.00   501.00   
rank        12904.0   6439.07   3720.23  0.0  120.06  3216.25  6441.50   
popularity  14478.0   7220.26   4170.08  0.0  136.77  3611.50  7225.50   
members     14478.0  22966.40  74981.36  0.0   29.00   245.00  1679.50   
favorites   14478.0    311.65   2615.55  0.0    0.00     0.00     2.00   
episodes    14478.0     11.31     43.44  0.0    0.00     1.00     1.00   

                 75%        99%        max  
score           7.06       8.46       10.0  
scored_by    3941.50  194311.41  1009477.0  
rank         9664.00   12742.97    12919.0  
popularity  10827.75   14300.23    14487.0  
members     10379.00  361159.94  1456378.0  
favorites      23.00    6780.55   106895.0  
episodes       12.00     104.00     1818.0  

score == 0: 354 | scored_by == 0: 308 | both: 

In [8]:
# Categorical columns
for col in ["type", "source", "status", "rating"]:
    print(f"\n{col}: {anime[col].nunique(dropna=False)} distinct values")
    print(anime[col].value_counts(dropna=False).head(12))


type: 7 distinct values
type
TV         4271
OVA        3612
Movie      2624
Special    1948
ONA        1144
Music       849
Unknown      30
Name: count, dtype: int64

source: 16 distinct values
source
Unknown         4210
Original        3368
Manga           3120
Visual novel     888
Game             607
Light novel      563
Other            408
Novel            359
Music            314
4-koma manga     224
Web manga        150
Picture book      96
Name: count, dtype: int64

status: 3 distinct values
status
Finished Airing     13791
Currently Airing      351
Not yet aired         336
Name: count, dtype: int64

rating: 7 distinct values
rating
PG-13 - Teens 13 or older         5020
G - All Ages                      4541
PG - Children                     1279
Rx - Hentai                       1219
R - 17+ (violence & profanity)     997
R+ - Mild Nudity                   878
None                               544
Name: count, dtype: int64


In [9]:
# Release year from aired_string (the first 4-digit number is the start year)
aired_year = pd.to_numeric(anime["aired_string"].str.extract(r"(\d{4})", expand=False), errors="coerce")
print("aired_string without a year:", aired_year.isna().sum(), f"({aired_year.isna().mean():.1%})")
print(aired_year.describe().round(1))
print(anime.loc[aired_year.isna(), "aired_string"].value_counts().head(5))
print("examples:", anime["aired_string"].dropna().head(5).tolist())

# Episode length in minutes from the text in `duration`
print("\n", anime["duration"].value_counts().head(8))
hours = pd.to_numeric(anime["duration"].str.extract(r"(\d+)\s*hr")[0], errors="coerce").fillna(0)
minutes = pd.to_numeric(anime["duration"].str.extract(r"(\d+)\s*min")[0], errors="coerce").fillna(0)
has_time = anime["duration"].str.contains(r"\d+\s*(?:hr|min)", na=False)
duration_min = (hours * 60 + minutes).where(has_time)
print("duration not parsable:", (~has_time).sum(), f"({(~has_time).mean():.1%})")
print(duration_min.describe(percentiles=[.01, .5, .99]).round(1))

aired_string without a year: 223 (1.5%)
count    14255.0
mean      2003.1
std         14.9
min       1917.0
25%       1997.0
50%       2008.0
75%       2014.0
max       2019.0
Name: aired_string, dtype: float64
aired_string
Not available    223
Name: count, dtype: int64
examples: ['Jan 13, 2012 to Mar 30, 2012', 'Apr 2, 2007 to Oct 1, 2007', 'Oct 4, 2008 to Sep 25, 2009', 'Aug 16, 2002 to May 23, 2003', 'Oct 6, 2012 to Mar 30, 2013']

 duration
24 min. per ep.    1513
25 min. per ep.     808
23 min. per ep.     797
30 min. per ep.     749
2 min.              476
Unknown             405
3 min.              371
4 min.              336
Name: count, dtype: int64
duration not parsable: 718 (5.0%)
count    13760.0
mean        26.8
std         25.5
min          1.0
1%           1.0
50%         24.0
99%        115.0
max        231.0
Name: 0, dtype: float64


In [10]:
# genre holds several values in one cell
genres = anime["genre"].dropna().str.split(", ")
print("anime without genre:", anime["genre"].isna().sum())
print("genres per anime:", genres.str.len().describe().round(2).to_dict())
print("distinct genres:", genres.explode().nunique())
print(genres.explode().value_counts().head(10))

# related: how are sequels stored, and how many anime have one?
print("\nexample:", anime["related"].dropna().head(2).tolist())
for relation in ["Sequel", "Prequel", "Side story", "Spin-off"]:
    share = anime["related"].str.contains(f"'{relation}'", na=False).mean()
    print(f"anime with a '{relation}' entry: {share:.1%}")

anime without genre: 64
genres per anime: {'count': 14414.0, 'mean': 2.91, 'std': 1.65, 'min': 1.0, '25%': 2.0, '50%': 3.0, '75%': 4.0, 'max': 13.0}
distinct genres: 43
genre
Comedy           5251
Action           3253
Fantasy          2703
Adventure        2582
Drama            2294
Sci-Fi           2272
Kids             2003
Shounen          1771
Romance          1634
Slice of Life    1532
Name: count, dtype: int64

example: ["{'Adaptation': [{'mal_id': 17207, 'type': 'manga', 'url': 'https://myanimelist.net/manga/17207/Inu_x_Boku_SS', 'title': 'Inu x Boku SS'}], 'Sequel': [{'mal_id': 13403, 'type': 'anime', 'url': 'https://myanimelist.net/anime/13403/Inu_x_Boku_SS_Special', 'title': 'Inu x Boku SS Special'}]}", "{'Adaptation': [{'mal_id': 759, 'type': 'manga', 'url': 'https://myanimelist.net/manga/759/Seto_no_Hanayome', 'title': 'Seto no Hanayome'}], 'Sequel': [{'mal_id': 4163, 'type': 'anime', 'url': 'https://myanimelist.net/anime/4163/Seto_no_Hanayome_OVA', 'title': 'Seto no Hanay

### Observations for part 2

**TODO (fill in after the run):**
- Placeholders and missingness: which columns are unusable or need a placeholder-to-`NaN` step?
- Implausible values: `episodes == 0`, extreme `duration`, release years outside 1900–2018.
- Reliability of `score`: how many anime have very few raters?
- Whether `related` is parsable enough for the sequel question, and whether `aired_string` and `duration` can be parsed for the time-trend and ML questions.

## 3. User table

In [11]:
count_cols = ["user_watching", "user_completed", "user_onhold", "user_dropped", "user_plantowatch"]
stat_cols = ["user_days_spent_watching", "stats_mean_score", "stats_rewatched", "stats_episodes"]

print(users.dtypes.value_counts().to_dict())
print(missing_overview(users).head(8))
print("access_rank non-missing:", users["access_rank"].notna().sum())

print("\ngender:")
print(users["gender"].value_counts(dropna=False))
print("\ndistinct locations:", users["location"].nunique(),
      "| missing:", users["location"].isna().sum())
print(users["location"].value_counts().head(10))

{dtype('O'): 6, dtype('int64'): 6, dtype('float64'): 5}
                  missing_count  missing_pct
access_rank              302675       100.00
location                 145876        48.20
birth_date               133926        44.25
gender                    84875        28.04
stats_episodes              129         0.04
join_date                   129         0.04
stats_rewatched             129         0.04
stats_mean_score            129         0.04
access_rank non-missing: 0

gender:
gender
Male          135181
NaN            84875
Female         81374
Non-Binary      1245
Name: count, dtype: int64

distinct locations: 55283 | missing: 145876
location
Poland         2425
Germany        1947
Canada         1821
California     1510
Brazil         1459
France         1135
Philippines    1077
Australia      1032
Sweden         1013
Finland         898
Name: count, dtype: int64


In [12]:
# Dates: placeholders and implausible values
for col in ["birth_date", "join_date", "last_online"]:
    parsed = pd.to_datetime(users[col], errors="coerce")
    print(f"{col:12s} unparsable/missing: {parsed.isna().sum():>7,}",
          "| '0000-00-00':", int((users[col] == "0000-00-00").sum()),
          "| earliest:", parsed.min(), "| latest:", parsed.max())
    users[col + "_dt"] = parsed

age_2018 = (pd.Timestamp("2018-01-01") - users["birth_date_dt"]).dt.days / 365.25
print("\nage at 2018-01-01:")
print(age_2018.describe(percentiles=[.01, .05, .5, .95, .99]).round(1))
print("age < 8:", int((age_2018 < 8).sum()), "| age > 80:", int((age_2018 > 80).sum()))
print("last_online before 2000:", int((users["last_online_dt"].dt.year < 2000).sum()),
      "| joined after last_online:", int((users["join_date_dt"] > users["last_online_dt"]).sum()))

# Aggregates reported per user
print(users[stat_cols].describe(percentiles=[.01, .5, .99, .999]).T.round(2))
print("stats_mean_score == 0:", int((users["stats_mean_score"] == 0).sum()))
print("users declaring no anime at all:", int((users[count_cols].sum(axis=1) == 0).sum()))

birth_date   unparsable/missing: 133,934 | '0000-00-00': 0 | earliest: 1800-01-01 00:00:00 | latest: 2137-01-01 00:00:00
join_date    unparsable/missing:     129 | '0000-00-00': 0 | earliest: 2004-11-05 00:00:00 | latest: 2018-05-24 00:00:00
last_online  unparsable/missing:     129 | '0000-00-00': 0 | earliest: 1900-01-01 00:13:00 | latest: 2018-05-25 12:53:00

age at 2018-01-01:
count    168741.0
mean         24.9
std           7.0
min        -119.0
1%           14.7
5%           17.6
50%          24.2
95%          33.0
99%          44.4
max         218.0
Name: birth_date_dt, dtype: float64
age < 8: 769 | age > 80: 955
last_online before 2000: 89283 | joined after last_online: 89283
                             count     mean        std  min   1%      50%  \
user_days_spent_watching  302675.0    69.08    4741.02  0.0  0.0    39.58   
stats_mean_score          302546.0     7.12       2.57  0.0  0.0     7.80   
stats_rewatched           302546.0    10.38      44.24  0.0  0.0     0.00   

### Observations for part 3

**TODO (fill in after the run):** which gender and location values need grouping or cleaning, how many ages are impossible, whether `last_online` and `join_date` look reliable, and whether any aggregate has extreme values (candidates for truncation or robust statistics).

## 4. Do the per-user counts agree with the list table?

`UserList` reports how many anime each user has in each status. `UserAnimeList` holds the actual rows. If both come from the same crawl, the counts should match. This also settles the 8 users in step 1 who declare anime but have no rows.

In [13]:
if RUN_LISTS:
    lists = pd.read_csv(
        paths["UserAnimeList.csv"], usecols=["username", "my_status"],
        dtype={"username": "category", "my_status": "int8"}, **na_kwargs,
    )
    print(f"list rows: {len(lists):,} | username NaN: {lists.username.isna().sum()}")

    per_status = lists.groupby(["username", "my_status"], observed=True).size().unstack(fill_value=0)
    print("usernames in lists but not in users:", len(set(per_status.index) - set(users.username)))

    status_to_col = {1: "user_watching", 2: "user_completed", 3: "user_onhold",
                     4: "user_dropped", 6: "user_plantowatch"}
    actual = per_status.reindex(columns=list(status_to_col)).fillna(0).astype(int)
    actual.columns = [c + "_actual" for c in status_to_col.values()]
    actual.index = actual.index.astype(object)
    declared = users.set_index("username")[count_cols].add_suffix("_declared")
    merged = declared.join(actual, how="left").fillna(0)
    print("merge rows:", len(merged), "| users:", len(users))

    for col in count_cols:
        diff = merged[col + "_actual"] - merged[col + "_declared"]
        print(f"{col:18s} exact match: {(diff == 0).mean():6.1%} | median diff: {diff.median():.0f}"
              f" | mean abs diff: {diff.abs().mean():.2f}")

    declared_total = merged[[c + "_declared" for c in count_cols]].sum(axis=1)
    actual_total = merged[[c + "_actual" for c in count_cols]].sum(axis=1)
    print("\ndeclare 0 but have rows:", int(((declared_total == 0) & (actual_total > 0)).sum()),
          "| declare > 0 but have no rows:", int(((declared_total > 0) & (actual_total == 0)).sum()))
else:
    print("Skipped (RUN_LISTS = False)")

list rows: 80,076,112 | username NaN: 0
usernames in lists but not in users: 0
merge rows: 302675 | users: 302675
user_watching      exact match:  98.2% | median diff: 0 | mean abs diff: 0.03
user_completed     exact match:  98.7% | median diff: 0 | mean abs diff: 0.07
user_onhold        exact match:  99.9% | median diff: 0 | mean abs diff: 0.01
user_dropped       exact match:  99.9% | median diff: 0 | mean abs diff: 0.01
user_plantowatch   exact match:  99.4% | median diff: 0 | mean abs diff: 0.04

declare 0 but have rows: 6 | declare > 0 but have no rows: 8


### Observations for part 4

**TODO (fill in after the run):** how often do declared and actual counts agree, in which direction do they differ, and how many users declare anime but have no rows? If the counts disagree widely, `UserList` aggregates should not be used as ground truth for the questions.

## 5. The users with no row in the list table: is the list table missing them?

In step 1 we found **19,630 users** who exist in `UserList` but own no row in `UserAnimeList`. For 19,622 of them the profile declares 0 anime, so an empty list is the natural explanation. But a declared count is only one column. If these users also have **other activity values** (days spent watching, episodes watched, a mean score), their list may exist on MyAnimeList and simply be missing from our list table.

| What we see among these users | What it means |
|---|---|
| `user_days_spent_watching`, `stats_episodes` and `stats_mean_score` are all 0 | The list is really empty. Nothing is missing |
| Many have positive values there | They watched and rated anime, so their list is **missing from the list table** (not crawled, private, or lost) |
| Their profile fields (gender, birth date, location) are filled as often as for other users | They are normal active profiles. If those fields are mostly empty, they are probably inactive accounts |

In [14]:
if RUN_LISTS:
    users["has_rows"] = users["username"].isin(actual.index)
    no_rows = users[~users["has_rows"]]
    with_rows = users[users["has_rows"]]
    declared_size = no_rows[count_cols].sum(axis=1)
    print(f"users without a row in the list table: {len(no_rows):,} | with rows: {len(with_rows):,}")
    print(f"  declared list size 0: {int((declared_size == 0).sum()):,} | declared list size > 0: {int((declared_size > 0).sum()):,}")

    print("\nactivity values of the users without rows (an empty account would show 0 everywhere):")
    for col in ["user_days_spent_watching", "stats_episodes", "stats_rewatched", "stats_mean_score"]:
        s = no_rows[col]
        print(f"  {col:26s} missing: {int(s.isna().sum()):>6,} | zero: {int((s == 0).sum()):>6,} | positive: {int((s > 0).sum()):>6,}"
              f" | max: {s.max():,.1f}")
    active = ((no_rows["user_days_spent_watching"] > 0) | (no_rows["stats_episodes"] > 0) | (no_rows["stats_mean_score"] > 0))
    print(f"\nusers without rows that show activity in at least one of the three columns: {int(active.sum()):,}")

    print("\nprofile fields filled (share of users), compared with the users that have rows:")
    fields = ["gender", "birth_date", "location", "join_date", "last_online"]
    print(pd.DataFrame({"no_rows": no_rows[fields].notna().mean(),
                        "with_rows": with_rows[fields].notna().mean()}).round(3).to_string())

    print("\nusers without rows that declare anime (the unexplained group):")
    cols_show = ["username"] + count_cols + ["user_days_spent_watching", "stats_episodes", "stats_mean_score"]
    print(no_rows.loc[declared_size > 0, cols_show].sort_values("user_completed", ascending=False).head(15).to_string(index=False))
else:
    print("Skipped (RUN_LISTS = False)")

users without a row in the list table: 19,630 | with rows: 283,045
  declared list size 0: 19,622 | declared list size > 0: 8

activity values of the users without rows (an empty account would show 0 everywhere):
  user_days_spent_watching   missing:      0 | zero: 19,624 | positive:      6 | max: 260.7
  stats_episodes             missing:      2 | zero: 19,593 | positive:     35 | max: 30,300.0
  stats_rewatched            missing:      2 | zero: 19,624 | positive:      4 | max: 112.0
  stats_mean_score           missing:      2 | zero: 19,593 | positive:     35 | max: 10.0

users without rows that show activity in at least one of the three columns: 47

profile fields filled (share of users), compared with the users that have rows:
             no_rows  with_rows
gender         0.311      0.748
birth_date     0.263      0.578
location       0.221      0.539
join_date      1.000      1.000
last_online    1.000      1.000

users without rows that declare anime (the unexplained group):


### Observations for part 5

**TODO (fill in after the run):** How many of the 19,630 users show activity (days, episodes or a mean score above 0)? Are the 19,622 users with a declared size of 0 also 0 in the three activity columns? Are their profile fields filled as often as for the users with rows? What do the 8 users who declare anime look like? Decide: are these users empty accounts (drop them from list analyses) or users whose list is missing (report how many)?

## 6. Clarity checks (NEW, added after the first run)

Parts 1 to 5 ran once. Some sentences of their observations rest on numbers that the first run did not print, and some hypotheses were never tested. This part prints them. It changes nothing in the data, and it reuses the objects built above (`anime`, `users`, `age_2018`, `merged`, ...), so run the notebook from the top.

| Check | What it settles |
|---|---|
| A. Anime values | How many `rank` and `popularity` are 0? Which words fill `broadcast`? What are the 313 `duration` texts that cannot be parsed? Which anime have `score = 0` but raters? |
| B. User dates and ages | Is `last_online` before 2000 always the year 1900 (our hypothesis)? Which `birth_date` texts cannot be parsed? Which birth years give the impossible ages? |
| C. Extremes | How many users lie above several caps of `stats_episodes` and `user_days_spent_watching`, and what do the largest ones look like? This helps to choose a cap in step 4. |
| D. `deactivated` usernames | How many usernames start with `deactivated`, and do they own list rows? (One of the 8 users who declare anime without rows is `deactivated-180120`.) |
| E. Declared against actual counts | In which direction do the counts differ, and who are the 6 users that declare 0 anime but have rows? |


In [15]:
# NEW A: anime values that the first run did not print
print("rank == 0:", int((anime["rank"] == 0).sum()), "| rank missing:", int(anime["rank"].isna().sum()),
      "| popularity == 0:", int((anime["popularity"] == 0).sum()))

print("\nbroadcast: the words that count as placeholders")
print(anime.loc[anime["broadcast"].isin(PLACEHOLDERS), "broadcast"].value_counts().to_string())

has_time_text = anime["duration"].str.contains(r"\d+\s*(?:hr|min)", na=False)
print("\nduration: the texts that cannot be parsed into minutes")
print(anime.loc[~has_time_text, "duration"].value_counts(dropna=False).head(10).to_string())

odd_score = (anime["score"] == 0) & (anime["scored_by"] > 0)
print("\nscore == 0 although scored_by > 0:", int(odd_score.sum()))
print(anime.loc[odd_score, ["anime_id", "title", "type", "status", "scored_by", "members"]]
      .sort_values("scored_by", ascending=False).head(10).to_string(index=False))


rank == 0: 9 | rank missing: 1574 | popularity == 0: 9

broadcast: the words that count as placeholders
broadcast
Unknown    2241

duration: the texts that cannot be parsed into minutes
duration
Unknown            405
30 sec.             54
30 sec. per ep.     42
15 sec.             30
15 sec. per ep.     17
40 sec. per ep.      8
45 sec.              7
39 sec. per ep.      6
52 sec.              5
16 sec.              5

score == 0 although scored_by > 0: 46
 anime_id                                                   title    type        status  scored_by  members
    37799                               Tokyo Ghoul:re 2nd Season      TV Not yet aired         15    28258
    37675                                            Overlord III      TV Not yet aired         12    60107
    35760                             Shingeki no Kyojin Season 3      TV Not yet aired         11   170892
    34134                                         One Punch Man 2      TV Not yet aired         10   234

In [16]:
# NEW B: user dates and ages
last_year = users["last_online_dt"].dt.year
print("last_online before 2000, by year:", last_year[last_year < 2000].value_counts().sort_index().to_dict())
print("last_online year counts from 2000 on:", last_year[last_year >= 2000].value_counts().sort_index().to_dict())

unparsable = users["birth_date"].notna() & users["birth_date_dt"].isna()
print("\nbirth_date texts that cannot be parsed:", users.loc[unparsable, "birth_date"].tolist())

birth_year = users["birth_date_dt"].dt.year
print("birth years of the users under 8 at 2018-01-01:", birth_year[age_2018 < 8].value_counts().head(8).to_dict())
print("birth years of the users over 80 at 2018-01-01:", birth_year[age_2018 > 80].value_counts().head(8).to_dict())


last_online before 2000, by year: {1900.0: 89283}
last_online year counts from 2000 on: {2007.0: 638, 2008.0: 5799, 2009.0: 11576, 2010.0: 14067, 2011.0: 12666, 2012.0: 12653, 2013.0: 17328, 2014.0: 15868, 2015.0: 18420, 2016.0: 24231, 2017.0: 40663, 2018.0: 39354}

birth_date texts that cannot be parsed: ['1-06-27', '1-04-22', '1-01-01', '1337-01-01', '1-01-01', '1337-01-01', '9001-12-31', '6969-01-01']
birth years of the users under 8 at 2018-01-01: {2012.0: 91, 2013.0: 87, 2011.0: 86, 2018.0: 82, 2014.0: 82, 2015.0: 79, 2010.0: 73, 2016.0: 71}
birth years of the users over 80 at 2018-01-01: {1930.0: 842, 1931.0: 41, 1934.0: 15, 1933.0: 14, 1937.0: 13, 1932.0: 13, 1935.0: 8, 1936.0: 8}


In [17]:
# NEW C: how many users lie above several caps, and what do the largest look like?
caps = {"stats_episodes": [10_000, 50_000, 100_000, 1_000_000],
        "user_days_spent_watching": [365, 1_000, 3_650, 36_500]}
for col, values in caps.items():
    print(col, {f"> {v:,}": int((users[col] > v).sum()) for v in values})

print()
show = ["username", "stats_episodes", "user_days_spent_watching", "user_completed", "user_plantowatch"]
print(users.nlargest(8, "stats_episodes")[show].to_string(index=False))


stats_episodes {'> 10,000': 19176, '> 50,000': 602, '> 100,000': 204, '> 1,000,000': 20}
user_days_spent_watching {'> 365': 1673, '> 1,000': 192, '> 3,650': 42, '> 36,500': 3}

       username  stats_episodes  user_days_spent_watching  user_completed  user_plantowatch
        warrose     243794881.0                2601351.02            2034                 7
      mmmapples      18218986.0                  23640.42               0                 0
        Kaiyuso      16791524.0                   2278.30             380               479
darkraigiratina      16776960.0                   8555.96               0                 2
 Kitayama_Naoki      10989789.0                   1376.93            2051               443
      sasuke423       5433345.0                    707.53               0                 1
          Fovez       3547267.0                  30197.12             802                 3
      Confucius       3287576.0                 165535.90             719              

In [18]:
# NEW D: usernames that start with "deactivated"
deactivated = users[users["username"].str.lower().str.startswith("deactivated")]
print("usernames starting with 'deactivated':", len(deactivated))
if "has_rows" in users:
    print("  of them with at least one list row:", int(deactivated["has_rows"].sum()))
print(deactivated[["username", "user_completed", "user_plantowatch", "user_days_spent_watching"]].head(10).to_string(index=False))


usernames starting with 'deactivated': 10
  of them with at least one list row: 6
             username  user_completed  user_plantowatch  user_days_spent_watching
      deactivatedshio             170                28                     74.16
     deactivated13041               0                 0                      0.00
     deactivated20325               0                 0                      0.00
       Deactivated453              13                26                      8.49
deactivated_yorozuuya              70                 0                     22.98
     deactivated30325               0                 0                      0.00
       deactivated000               7                 6                      2.73
   deactivated-180120             262                88                    108.67
       deactivated106             715               422                    205.91
          Deactivated              56                22                     47.28


In [19]:
# NEW E: declared against actual counts, direction of the differences
if RUN_LISTS:
    for col in count_cols:
        d = merged[col + "_actual"] - merged[col + "_declared"]
        print(f"{col:18s} actual > declared: {int((d > 0).sum()):>6,} | actual < declared: {int((d < 0).sum()):>6,}"
              f" | largest gap: {int(d.abs().max()):,}")

    zero_declared = merged[(declared_total == 0) & (actual_total > 0)]
    print("\nusers that declare 0 anime but have rows:")
    print(zero_declared[[c + "_actual" for c in count_cols]].assign(total_rows=actual_total[zero_declared.index]).to_string())
else:
    print("Skipped (RUN_LISTS = False)")


user_watching      actual > declared:  2,954 | actual < declared:  2,510 | largest gap: 255
user_completed     actual > declared:    329 | actual < declared:  3,482 | largest gap: 1,043
user_onhold        actual > declared:     92 | actual < declared:    171 | largest gap: 215
user_dropped       actual > declared:     93 | actual < declared:    171 | largest gap: 208
user_plantowatch   actual > declared:  1,488 | actual < declared:    391 | largest gap: 1,432

users that declare 0 anime but have rows:
                  user_watching_actual  user_completed_actual  user_onhold_actual  user_dropped_actual  user_plantowatch_actual  total_rows
username                                                                                                                                   
ker0chan                          51.0                  574.0                 0.0                  0.0                   1432.0      2057.0
Sora-Senpai                        0.0                   93.0            

### Observations for the clarity checks

**TODO (fill in after the run):**
- A: how many `rank` and `popularity` are 0? Which words fill `broadcast`, and which texts make `duration` unparsable? Are the anime with `score = 0` and raters few or many?
- B: is every `last_online` before 2000 in the year 1900? If yes, the hypothesis that the year was lost is supported. Which birth years give the impossible ages?
- C: at which cap do few enough users lie above it? Are the largest users plausible?
- D: how many usernames start with `deactivated`, and do they own rows?
- E: is the actual count usually larger or smaller than the declared one? Who are the 6 users that declare 0 but have rows?
